# Aula 5 · Sua primeira rede neural

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlexChequer/notebooks-insperai/blob/main/trainees/aula-05-backpropagation.ipynb)

**Trilha de Trainees — InsperAI** · o par prático da [Aula 5](https://trilhas-insperai.vercel.app/trainees/aulas/aula-05/)

---

Você vai montar e treinar uma rede neural do começo ao fim.

**O problema:** prever se um aluno passa na prova sabendo quantas horas ele estudou e quantas horas dormiu na noite anterior.

**Como funciona:** as células prontas você só roda. Onde estiver escrito `TODO`, você completa, trocando o `...` pelo código.

## 0. Os dados

200 alunos inventados. Cada um tem duas informações (horas de estudo e horas de sono) e uma resposta: passou (1) ou não passou (0).

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
rng = np.random.default_rng(0)

estudo = rng.uniform(0, 10, 200)   # horas de estudo
sono = rng.uniform(3, 10, 200)     # horas de sono
passou = (estudo >= 4) & (sono >= 6)

# A rede aprende melhor com números pequenos, então dividimos as horas por 10.
X = torch.tensor(np.column_stack([estudo, sono]) / 10, dtype=torch.float32)
y = torch.tensor(passou, dtype=torch.float32).reshape(-1, 1)

print("entradas:", X.shape, "  respostas:", y.shape)
print("passaram:", int(y.sum()), "de", len(y))

plt.scatter(estudo[passou], sono[passou], c="#10B981", label="passou")
plt.scatter(estudo[~passou], sono[~passou], c="#EF4444", label="não passou")
plt.xlabel("horas de estudo")
plt.ylabel("horas de sono")
plt.legend()
plt.show()

Repare no gráfico: estudar muito não basta, quem dormiu pouco também não passa. Uma reta só não separa os verdes dos vermelhos. É por isso que precisamos de uma rede com camada escondida.

## Tarefa 1 · Monte a rede

A rede tem três partes:

- **entrada:** 2 números (estudo e sono)
- **camada escondida:** 8 neurônios com ReLU
- **saída:** 1 neurônio com sigmoide, que devolve a probabilidade de o aluno passar (entre 0 e 1)

Dica: é igual à Aula 4, só que com uma `nn.Sigmoid()` no final.
`nn.Sequential(nn.Linear(entradas, escondidos), nn.ReLU(), nn.Linear(escondidos, saidas), nn.Sigmoid())`

In [ ]:
rede = nn.Sequential(
    # TODO: Linear de 2 para 8
    # TODO: ReLU
    # TODO: Linear de 8 para 1
    # TODO: Sigmoid
)

print(rede)
print("previsão para os 3 primeiros alunos:", rede(X[:3]).detach().flatten())

Se apareceram 3 números entre 0 e 1, a rede está montada. Por enquanto ela chuta, porque os pesos são aleatórios.

## Tarefa 2 · Treine a rede

Treinar é repetir o mesmo ciclo muitas vezes:

1. **Ida:** a rede faz a previsão para todos os alunos.
2. **Perda:** um número que mede o quanto a previsão errou.
3. **Volta (backpropagation):** `perda.backward()` calcula a culpa de cada peso pelo erro.
4. **Passo (gradient descent):** `otimizador.step()` ajusta cada peso um pouco contra a sua culpa.

O backpropagation só **calcula** a culpa. Quem **ajusta** os pesos é o passo.

Complete as linhas do ciclo dentro da função.

In [ ]:
def treinar(rede, passos=2000):
    funcao_perda = nn.BCELoss()                               # a perda da classificação binária (Aula 3)
    otimizador = torch.optim.SGD(rede.parameters(), lr=1.0)   # o gradient descent
    historico = []

    for passo in range(passos):
        previsao = ...      # TODO 1 (ida): passe X pela rede
        perda = ...         # TODO 2 (perda): funcao_perda(previsao, y)

        otimizador.zero_grad()   # apaga a culpa do passo anterior (pronto)
        ...                 # TODO 3 (volta): calcule a culpa de cada peso
        ...                 # TODO 4 (passo): ajuste os pesos

        historico.append(perda.item())

    return historico


historico = treinar(rede)
print(f"perda no começo: {historico[0]:.3f}   perda no fim: {historico[-1]:.3f}")

plt.plot(historico)
plt.xlabel("passo")
plt.ylabel("perda")
plt.title("A perda caindo durante o treino")
plt.show()

## Tarefa 3 · Use a rede

Primeiro, quantos alunos a rede acerta? E onde ela aprendeu a dividir quem passa de quem não passa? Só rode.

In [ ]:
def acuracia(rede):
    acertos = ((rede(X) > 0.5).float() == y).float().mean()
    return acertos.item()


def desenhar_fronteira(rede):
    ee, ss = np.meshgrid(np.linspace(0, 10, 200), np.linspace(3, 10, 200))
    grade = torch.tensor(np.column_stack([ee.ravel(), ss.ravel()]) / 10, dtype=torch.float32)
    prob = rede(grade).detach().numpy().reshape(ee.shape)

    plt.contourf(ee, ss, prob, levels=[0, 0.5, 1], colors=["#FEE2E2", "#D1FAE5"])
    plt.scatter(estudo[passou], sono[passou], c="#10B981", label="passou")
    plt.scatter(estudo[~passou], sono[~passou], c="#EF4444", label="não passou")
    plt.xlabel("horas de estudo")
    plt.ylabel("horas de sono")
    plt.legend()
    plt.show()


print(f"acurácia: {acuracia(rede):.0%}")
desenhar_fronteira(rede)

O fundo verde é onde a rede diz "passa", o vermelho é onde diz "não passa". Ninguém escreveu a regra para ela: a rede descobriu sozinha, repetindo ida, perda, volta e passo.

Agora pergunte para a rede: um aluno que estudou **6 horas** e dormiu **8 horas** passa? E um que estudou **9 horas** e dormiu só **4**?

Lembre de dividir as horas por 10, como fizemos com os dados.

In [ ]:
aluno = torch.tensor([[6 / 10, 8 / 10]])
prob = ...   # TODO: passe o aluno pela rede
print(f"6h de estudo, 8h de sono: {prob.item():.0%} de chance de passar")

# TODO: faça o mesmo para o aluno que estudou 9 horas e dormiu 4

---
## Desafios

1. **Sem ReLU.** Monte uma rede igual, mas sem a `nn.ReLU()`, treine com `treinar(...)` e desenhe a fronteira. O que mudou no formato dela?
2. **Menos neurônios.** Troque os 8 neurônios escondidos por 1. A rede ainda acerta tanto?
3. **Sem a volta.** Copie a função `treinar`, apague a linha do `perda.backward()` e treine uma rede nova com ela. A perda ainda cai? Por quê?

In [ ]:
# Use esta célula para os desafios